# Normalised total energy for a set of runs

One notebook for what used to be two (`EnergyPlotsforDifferentBeta`, `EnergyPlotsforDifferentPpc`, identical apart from
the file list and labels; they are in `archive/quality_checks_superseded/`). Choose the runs with `DATASET`:

* `"beta"`: the two extreme-beta runs (beta_p = 16 and 0.0625)
* `"ppc"`: the four particles-per-cell runs (2048, 1024, 512, 256 ppc; eta = 2.5e-4)

The figure is saved as `EnergyPlot_<DATASET>.png` (both old notebooks saved `EnergyPlotPpc.png`, so they overwrote
each other). The last cell, |B_perp|^2 maps of the four resistivity runs at t = 250, was in both notebooks.

In [ ]:
#All the imports
import h5py
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d
from scipy.stats import norm
from scipy.stats import kurtosis
from scipy.ndimage import maximum_filter, label
import multiprocessing as mp
import os

# shared code and paths (see README.md and shared/)
import sys
from pathlib import Path
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "shared" / "paths.py").exists())
sys.path.insert(0, str(REPO))
from shared.paths import EXTERNAL_DATA_ROOT, results_dir
RESULTS = results_dir("quality_checks", "EnergyPlots")

In [ ]:
DATASET = "ppc"      # "beta" or "ppc"

# runs under EXTERNAL_DATA_ROOT/DEVESH/ (the /DATA disk) and their legend labels
DATASETS = {
    "beta": dict(runs=["ExtremeBetaSimulations/16", "ExtremeBetaSimulations/0.0625"],
                 labels=["16", "0.0625"]),
    "ppc":  dict(runs=["ResistivityCheck/2.5e-4_2048", "ResistivityCheck/2.5e-4_1024",
                       "ResistivityCheck/2.5e-4_512", "ResistivityCheck/2.5e-4_256"],
                 labels=["2048 ppc", "1024 ppc", "512 ppc", "256 ppc"]),
}

In [ ]:
# --- File paths ---
runs = DATASETS[DATASET]["runs"]
File_Etot = [f"{EXTERNAL_DATA_ROOT}/DEVESH/{run}/EnTot_ApJ_.h5" for run in runs]
File_Time = [f"{EXTERNAL_DATA_ROOT}/DEVESH/{run}/Time_ApJ_.h5" for run in runs]

labels = DATASETS[DATASET]["labels"]

# Helper function to load DS1 from a file
def load_dataset(fp):
    with h5py.File(fp, "r") as f:
        key = list(f.keys())[0]        # get first dataset name, e.g. 'DS1'
        return f[key][:]

# --- Load all runs ---
energies = []
times = []
for efile, tfile in zip(File_Etot, File_Time):
    energies.append(load_dataset(efile))
    times.append(load_dataset(tfile))

# --- Normalize energies ---
normalized = [E / E[0] for E in energies]

# --- Plot ---
plt.figure(figsize=(8, 5))
plt.xlim(0, 300) 
plt.ylim(0.98, 1.02)

for t, Enorm, label in zip(times, normalized, labels):
    plt.plot(t, Enorm, label=label)

plt.xlabel("Time")
plt.ylabel("E / E₀")
plt.title("Normalized Total Energy vs Time")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(RESULTS / f"EnergyPlot_{DATASET}.png", dpi=300)
plt.show()

In [ ]:
# Simulation directories
sim_dirs = [
    f'{EXTERNAL_DATA_ROOT}/DEVESH/ResistivityCheck/0.625e-4',
    f'{EXTERNAL_DATA_ROOT}/DEVESH/ResistivityCheck/1.25e-4',
    f'{EXTERNAL_DATA_ROOT}/DEVESH/ResistivityCheck/2.5e-4_2048',
    f'{EXTERNAL_DATA_ROOT}/DEVESH/ResistivityCheck/5e-4'
]

dx = 0.0625
dy = 0.0625

for sim_path in sim_dirs:
    # Identify simulation label (last folder name)
    sim_label = os.path.basename(sim_path)

    # File paths
    File_Bx = os.path.join(sim_path, 'Bx_ApJ_t250.h5')
    File_By = os.path.join(sim_path, 'By_ApJ_t250.h5')

    # Read data
    with h5py.File(File_Bx, 'r') as fBx, h5py.File(File_By, 'r') as fBy:
        data_Bx = fBx['DS1'][:].T
        data_By = fBy['DS1'][:].T

    # Compute B_perp^2
    data_Bp = data_Bx**2 + data_By**2

    # Plot
    plt.figure(figsize=(10, 8))
    img = plt.imshow(data_Bp, cmap='plasma', vmax=0.5,
                     extent=[0, data_Bp.shape[1]*dx, 0, data_Bp.shape[0]*dy],
                     origin='lower')

    plt.colorbar(img)
    plt.title(r'$|B_\perp|^2$' + f"  (sim: {sim_label})")
    plt.xlabel(r'$x/d_{p}$')
    plt.ylabel(r'$y/d_{p}$')

    plt.tight_layout()
    outname = RESULTS / f"B_{sim_label}.png"
    plt.savefig(outname, dpi=300)
    plt.close()

    print(f"Saved: {outname}")